# Explore and prepare animal images

## 1. Objective and data source
Prepare Cat, Dog, Cow, Horse and Sheep images from Alessio Corrado's [Animals-10 dataset](https://www.kaggle.com/datasets/alessiocorrado99/animals10), version 2. The original collection has ten Italian folder labels. We keep five classes and record every retained image.

The Kaggle metadata lists GPL-2.0. Its description says the photographs were collected from Google Images and does not give individual credits. The original data and image-bearing outputs stay local. See `outputs/dataset_metadata.json` and `outputs/dataset_license.md`.

Download from the project folder with the official client: `.venv/bin/kaggle datasets download -d alessiocorrado99/animals10 -p data/raw --unzip`. This was actually run for this project.

## 2. Libraries and their roles
Pillow opens, checks and resizes images. NumPy stores pixels as arrays. pandas organizes labels, hashes and splits. Matplotlib draws distributions and image grids. ImageHash finds visually similar files. scikit-learn creates stratified groups so related images stay together. hashlib identifies exact files and identical decoded pixels.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd().parent
assert (ROOT / 'common.py').exists(), 'Open this notebook from the project or notebooks folder.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from common import LABELS, SEED, pixels
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 3. Understand the original labels
Count the files before filtering. Raw files are kept unchanged.

In [ ]:
from PIL import Image, ImageOps
import hashlib, imagehash
raw = ROOT / 'data/raw/raw-img'
assert raw.exists(), 'Download and unzip Animals-10 first.'
translation = {'cane':'Dog','cavallo':'Horse','elefante':'Elephant','farfalla':'Butterfly',
               'gallina':'Chicken','gatto':'Cat','mucca':'Cow','pecora':'Sheep',
               'ragno':'Spider','scoiattolo':'Squirrel'}
counts = pd.DataFrame([{'original_label':p.name,'english_label':translation[p.name],
                       'images':len([f for f in p.iterdir() if f.is_file()])}
                      for p in sorted(raw.iterdir()) if p.is_dir()])
counts.to_csv(ROOT / 'outputs/original_class_counts.csv', index=False)
display(counts)
fig, ax = plt.subplots(figsize=(9,3.5))
ax.bar(counts.english_label, counts.images, color='#38678d')
ax.set(ylabel='Images',title='Original Animals-10 class distribution')
ax.tick_params(axis='x', rotation=30); fig.tight_layout()
fig.savefig(ROOT / 'outputs/original_distribution.png',dpi=180); plt.show()

## 4. Check image integrity and exact duplicates
We verify each selected file and decode it fully. SHA-256 catches byte-for-byte duplicates. A second hash of oriented RGB pixels also catches identical images with different file metadata. We remove repeated copies, and exclude any exact duplicate that has conflicting class labels. A 64-bit perceptual hash of each image and its horizontal reflection helps find related images.

In [ ]:
records, corrupt = [], []
for directory in sorted(raw.iterdir()):
    if not directory.is_dir() or translation[directory.name] not in LABELS: continue
    for path in sorted(directory.iterdir()):
        if not path.is_file(): continue
        relative = path.relative_to(ROOT).as_posix()
        try:
            with Image.open(path) as image: image.verify()
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert('RGB'); image.load()
                width, height = image.size
                content = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
                phash = str(imagehash.phash(image))
                mirror = str(imagehash.phash(ImageOps.mirror(image)))
            records.append(dict(image_id=hashlib.sha256(relative.encode()).hexdigest()[:20],path=relative,
                original_label=directory.name,label=translation[directory.name],width=width,height=height,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest(),pixel_sha256=content,phash=phash,mirror_phash=mirror))
        except Exception as error:
            corrupt.append({'path':relative,'reason':str(error)})
all_images = pd.DataFrame(records)
all_images.to_csv(ROOT / 'outputs/image_audit.csv', index=False)
pd.DataFrame(corrupt,columns=['path','reason']).to_csv(ROOT / 'outputs/corrupt_images.csv', index=False)
conflicts = all_images.groupby('pixel_sha256').label.nunique()
conflict_hashes = set(conflicts[conflicts > 1].index)
clean = all_images[~all_images.pixel_sha256.isin(conflict_hashes)].drop_duplicates('pixel_sha256').copy().reset_index(drop=True)
removed = all_images[~all_images.image_id.isin(clean.image_id)].copy()
removed.to_csv(ROOT / 'outputs/exact_duplicates_removed.csv', index=False)
print('Selected files:',len(records)+len(corrupt),'Corrupted:',len(corrupt),
      'Exact duplicates or conflicting copies removed:',len(removed),'Retained:',len(clean))

## 5. Find related images before splitting
A perceptual-hash distance of at most six bits is a conservative candidate rule, not proof that images are duplicates. We group every candidate pair, including reflected versions, and keep each connected group in one split. We retain near-duplicate images rather than deleting uncertain matches. Contact sheets support visual review; no model scores guide cleaning.

This check cannot guarantee that every crop or related photograph is found. Missing source identifiers are a limitation of the dataset.

In [ ]:
# A small BK-tree finds nearby hashes without storing every pairwise distance.
class HashTree:
    def __init__(self): self.root = None
    def add(self, value, index):
        if self.root is None: self.root=[value,[index],{}]; return
        node=self.root
        while True:
            distance=(value ^ node[0]).bit_count()
            if distance==0: node[1].append(index); return
            if distance not in node[2]: node[2][distance]=[value,[index],{}]; return
            node=node[2][distance]
    def query(self,value,radius):
        stack=[] if self.root is None else [self.root]
        while stack:
            node=stack.pop(); distance=(value ^ node[0]).bit_count()
            if distance<=radius:
                for index in node[1]: yield index,distance
            stack.extend(child for d,child in node[2].items() if distance-radius<=d<=distance+radius)

parent=list(range(len(clean)))
def find(i):
    while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
    return i
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[max(a,b)]=min(a,b)
tree=HashTree(); candidates=[]
for i,row in clean.iterrows():
    matches={}
    for value in [int(row.phash,16),int(row.mirror_phash,16)]:
        for j,d in tree.query(value,6): matches[j]=min(d,matches.get(j,65))
    for j,d in matches.items():
        union(i,j)
        candidates.append({'left_id':clean.iloc[j].image_id,'right_id':row.image_id,'distance':d,
                           'left_label':clean.iloc[j].label,'right_label':row.label})
    tree.add(int(row.phash,16),i)
clean['group_id']=[clean.iloc[find(i)].image_id for i in range(len(clean))]
pairs=pd.DataFrame(candidates,columns=['left_id','right_id','distance','left_label','right_label'])
pairs.to_csv(ROOT / 'outputs/near_duplicate_candidates.csv',index=False)
print('Candidate pairs:',len(pairs),'Non-singleton groups:',int((clean.groupby('group_id').size()>1).sum()))
# Save all candidate pairs in readable local contact sheets.
private=ROOT / 'outputs/private'; private.mkdir(exist_ok=True)
lookup=clean.set_index('image_id')
for start in range(0,len(pairs),12):
    page=pairs.iloc[start:start+12]
    fig,axes=plt.subplots(len(page),2,figsize=(7,2.2*len(page)),squeeze=False)
    for axes_row,(_,pair) in zip(axes,page.iterrows()):
        for ax,key in zip(axes_row,['left_id','right_id']):
            r=lookup.loc[pair[key]]; ax.imshow(pixels(ROOT / r.path,160)); ax.axis('off')
            ax.set_title(f'{r.label} | {r.name[:8]} | distance {pair.distance}',fontsize=9)
    fig.tight_layout(); fig.savefig(private / f'near_pairs_{start//12+1:03d}.jpg',dpi=90); plt.close(fig)

## 6. Create stratified train validation and test splits
The target proportions are 70%, 15% and 15%, with seed 42. We use 20 stratified group folds: 14 for training, three for validation and three for testing. Whole groups make the proportions approximate at image level. We save the actual counts. No group, file hash or pixel hash may occur in more than one split.

Training fits model parameters. Validation checks model choices and CNN early stopping. Testing is reserved for final evaluation. Stratification keeps the animal proportions similar across splits.

Visual review found a horse photograph in the Cow folder. Its ID and exclusion reason are saved in `outputs/manual_review_decisions.csv`. The other four cross-label pairs are different animals with similar hash patterns; we retain their conservative grouping.

In [ ]:
# Apply the documented visual review before constructing the final splits.
review_path=ROOT / 'outputs/manual_review_decisions.csv'
review=pd.read_csv(review_path)
excluded_ids=set(review.loc[review.action.eq('exclude'),'image_id'])
clean=clean[~clean.image_id.isin(excluded_ids)].reset_index(drop=True)
print('Images excluded after visual review:',len(excluded_ids))
from sklearn.model_selection import StratifiedGroupKFold
splitter=StratifiedGroupKFold(n_splits=20,shuffle=True,random_state=SEED)
folds=np.empty(len(clean),dtype=int)
for fold,(_,idx) in enumerate(splitter.split(clean,clean.label,clean.group_id)): folds[idx]=fold
clean['split']=np.where(folds<14,'train',np.where(folds<17,'validation','test'))
clean['label_id']=clean.label.map({label:i for i,label in enumerate(LABELS)})
for column in ['group_id','sha256','pixel_sha256']:
    assert clean.groupby(column).split.nunique().max()==1, f'Leakage in {column}'
assert clean.image_id.is_unique
clean.to_csv(ROOT / 'data/processed/split_manifest.csv',index=False)
clean.to_csv(ROOT / 'outputs/split_manifest.csv',index=False)
distribution=pd.crosstab(clean.label,clean.split).reindex(LABELS)
distribution.to_csv(ROOT / 'outputs/split_class_counts.csv')
display(distribution)
print(clean.split.value_counts(normalize=True).round(4))
fig,ax=plt.subplots(figsize=(8,4)); distribution[['train','validation','test']].plot.bar(ax=ax)
ax.set(ylabel='Images',xlabel='Animal',title='Retained images in each split'); ax.tick_params(axis='x',rotation=0)
fig.tight_layout(); fig.savefig(ROOT / 'outputs/split_distribution.png',dpi=180); plt.show()

## 7. Resize and normalize the images
Both approaches use oriented RGB images and Pillow bilinear resizing. Logistic Regression uses 24 by 24 pixels flattened into 1,728 features. The CNN uses 64 by 64 pixels and keeps the spatial layout. These different input resolutions are a comparison limitation. We divide pixel intensities by 255 to obtain values between zero and one. The Logistic Regression pipeline learns any additional scaling from training folds only.

In [ ]:
# Store compact uint8 arrays; normalize when loading them for training.
X24=np.stack([pixels(ROOT / p,24) for p in clean.path])
X64=np.stack([pixels(ROOT / p,64) for p in clean.path])
np.savez_compressed(ROOT / 'data/processed/images.npz',pixels24=X24,pixels64=X64,
                    image_ids=clean.image_id.to_numpy(dtype=str))
config={'seed':SEED,'labels':LABELS,'label_mapping':{str(i):s for i,s in enumerate(LABELS)},
        'logistic_size':[24,24],'cnn_size':[64,64],'color_mode':'RGB','orientation':'EXIF transpose',
        'resize':'Pillow bilinear, direct square resize','normalization':'float32 / 255',
        'near_duplicate_rule':'64-bit pHash distance <= 6 including horizontal reflection',
        'split_method':'StratifiedGroupKFold 20 folds; train 0-13, validation 14-16, test 17-19'}
(ROOT / 'models/preprocessing.json').write_text(json.dumps(config,indent=2))
summary={'original_images':int(counts.images.sum()),'selected_files':len(records)+len(corrupt),
         'corrupted':len(corrupt),'exact_removed':len(removed),'manual_excluded':len(excluded_ids),'retained':len(clean),
         'near_pairs':len(pairs),'groups':clean.group_id.nunique(),
         'split_counts':clean.split.value_counts().to_dict()}
(ROOT / 'outputs/data_summary.json').write_text(json.dumps(summary,indent=2))
fig,axes=plt.subplots(2,5,figsize=(11,5))
for col,label in enumerate(LABELS):
    sample=clean[clean.label==label].sample(2,random_state=SEED)
    for row,(_,r) in enumerate(sample.iterrows()):
        axes[row,col].imshow(pixels(ROOT / r.path,160)); axes[row,col].axis('off'); axes[row,col].set_title(label)
fig.tight_layout(); fig.savefig(ROOT / 'outputs/private/animal_samples.png',dpi=160); plt.show()
print('Pixel arrays:',X24.shape,X64.shape,'Normalized range:',X24.min()/255,X24.max()/255)

## 8. Observations and limits
The five selected folders contain 12,840 images. No corrupted files or exact pixel copies were found. Visual review excluded one horse wrongly stored in the Cow folder, leaving 12,839 images. The saved split has 8,986 training, 1,928 validation and 1,925 test images.

There are 1,108 candidate near-duplicate pairs. Review covered all five cross-label pairs and a fixed-seed sample of 12 pairs across hash distances 0, 2, 4 and 6. Detected related images stay in one split. Perceptual hashing and sample review cannot certify every label or find every related source image.